#Задание 1. Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

 Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

In [38]:
import os
import sys
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
import pyspark.sql.types as t
from math import sqrt

# Задаем переменные окружения для корректной работы некоторых функций
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [39]:
# Задаем переменную окружения для парсинга xml
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.12:0.17.0 pyspark-shell'

os.chdir('../')

In [40]:
python_path = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LR2")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .getOrCreate()
)
spark

In [41]:
posts = spark.read \
    .format('xml') \
    .option('rowTag', 'row') \
    .load('posts_sample.xml')

print(f"Постов: {posts.count()}")


Постов: 46006


In [42]:
languages = spark.read \
    .option('header', 'true') \
    .csv('programming-languages.csv')

print(f"Языков: {languages.count()}")


Языков: 700


In [43]:
def show_info(data):

    print("\033[1mПервые 5 элементов\033[0m")
    data.show(n = 5)

    print("\033[1mКоличество элементов\033[0m")
    print(data.count())

In [44]:
show_info(posts)

Первые 5 элементов
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|_Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+---

In [45]:
posts_filtered = posts.select(
    F.year(F.to_timestamp("_CreationDate")).alias("year"),
    F.lower(F.col("_Tags")).alias("tags")
).filter(
    (F.col("year") >= 2010) & (F.col("year") <= 2020) & F.col("tags").isNotNull()
)


In [46]:
show_info(posts_filtered)

Первые 5 элементов
+----+--------------------+
|year|                tags|
+----+--------------------+
|2010|<c++><character-e...|
|2010|<sharepoint><info...|
|2010|<iphone><app-stor...|
|2010|<symfony1><schema...|
|2010|              <java>|
+----+--------------------+
only showing top 5 rows
Количество элементов
17642


In [47]:
languages_clean = languages.select(
    F.lower(F.trim("name")).alias("tag")
).dropDuplicates()

In [48]:
show_info(languages_clean)

Первые 5 элементов
+--------+
|     tag|
+--------+
|  ceylon|
|    hope|
|      m#|
|metafont|
| mortran|
+--------+
only showing top 5 rows
Количество элементов
698


Парсинг тегов поста
- regexp_replace - заменяет символы <> на пробелы
- split - разделяет строку на части по пробелам
- explode - преобразует массив в отдельные строки

In [49]:
posts_tags = posts_filtered.withColumn(
    "tag",
    F.explode(F.split(F.regexp_replace("tags", r"[<>]", " "), r"\s+"))
).filter(F.col("tag") != "")


In [50]:
show_info(posts_tags)

Первые 5 элементов
+----+--------------------+------------------+
|year|                tags|               tag|
+----+--------------------+------------------+
|2010|<c++><character-e...|               c++|
|2010|<c++><character-e...|character-encoding|
|2010|<sharepoint><info...|        sharepoint|
|2010|<sharepoint><info...|          infopath|
|2010|<iphone><app-stor...|            iphone|
+----+--------------------+------------------+
only showing top 5 rows
Количество элементов
52118


Объединяем две таблицы в одну

In [51]:
mentions = posts_tags.join(languages_clean, "tag", "inner") \
    .select("year", "tag")


In [52]:
show_info(mentions)

Первые 5 элементов
+----+----+
|year| tag|
+----+----+
|2010|java|
|2010| php|
|2010|ruby|
|2010|   c|
|2010| php|
+----+----+
only showing top 5 rows
Количество элементов
8054


In [53]:
stats = mentions.groupBy("year", "tag").count()

# данные разбиваются на группы по годам и обрабатываются уже отдельно в каждом окне
window = Window.partitionBy("year").orderBy(F.desc("count"), F.asc("tag"))

# внутри каждой партиции каждой строке присваивается ранк, берем значения для топ-10
top10 = stats.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") <= 10) \
    .orderBy("year", "rank")

top10.show(110)

+----+-----------+-----+----+
|year|        tag|count|rank|
+----+-----------+-----+----+
|2010|       java|   52|   1|
|2010|        php|   46|   2|
|2010| javascript|   44|   3|
|2010|     python|   26|   4|
|2010|objective-c|   23|   5|
|2010|          c|   20|   6|
|2010|       ruby|   12|   7|
|2010|     delphi|    8|   8|
|2010|applescript|    3|   9|
|2010|       bash|    3|  10|
|2011|        php|  102|   1|
|2011|       java|   93|   2|
|2011| javascript|   83|   3|
|2011|     python|   37|   4|
|2011|objective-c|   34|   5|
|2011|          c|   24|   6|
|2011|       ruby|   20|   7|
|2011|       perl|    9|   8|
|2011|     delphi|    8|   9|
|2011|       bash|    7|  10|
|2012|        php|  154|   1|
|2012| javascript|  132|   2|
|2012|       java|  124|   3|
|2012|     python|   69|   4|
|2012|objective-c|   45|   5|
|2012|          c|   27|   6|
|2012|       ruby|   27|   7|
|2012|       bash|   10|   8|
|2012|          r|    9|   9|
|2012|        lua|    6|  10|
|2013| jav

# Задание 2. Получившийся отчёт сохранить в формате Apache Parquet.

In [54]:
top10.write.mode("overwrite").parquet("report.parquet")

In [55]:
import shutil
from google.colab import files

# Архивируем папку
shutil.make_archive("report_parquet", "zip", "/content/report.parquet")
files.download("report_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>